# 6. Modelado con PySpark (MLlib)

En este capítulo se implementa el modelo de clasificación utilizando PySpark MLlib con el fin de evaluar la capacidad predictiva del algoritmo Random Forest sobre un conjunto de datos de gran escala.

Se realiza la construcción de la variable objetivo, el preprocesamiento de variables numéricas y categóricas, la creación de un pipeline de transformación y el entrenamiento del modelo. Finalmente, se evalúa el desempeño utilizando métricas de clasificación como AUC, accuracy, precision, recall y F1-score.

In [1]:
import os
import sys
import subprocess

# Esto intenta ubicar el java del env conda
java_path = os.path.join(sys.prefix, "Library", "bin", "java.exe")
if os.path.exists(java_path):
    os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "Library")
    os.environ["PATH"] = os.path.join(sys.prefix, "Library", "bin") + ";" + os.environ["PATH"]

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode("utf-8"))

JAVA_HOME = c:\Users\PC\miniconda3\envs\jb_lending\Library
openjdk version "17.0.18" 2026-01-20 LTS
OpenJDK Runtime Environment Zulu17.64+17-CA (build 17.0.18+8-LTS)
OpenJDK 64-Bit Server VM Zulu17.64+17-CA (build 17.0.18+8-LTS, mixed mode, sharing)



## 6.1 Inicialización de SparkSession

Se crea la sesión de Spark para trabajar con PySpark y ejecutar el flujo de modelado de forma distribuida.

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LendingClub_Default_PySpark")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark

## 6.2 Carga del dataset

Se carga el dataset de LendingClub en formato CSV. Debido al tamaño, se trabaja con un subconjunto de columnas relevantes y se activa el manejo de comillas y escapes para evitar errores de parseo.

In [5]:
from pyspark.sql.functions import col

DATA_PATH = "../data/accepted_2007_to_2018Q4.csv"  

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(DATA_PATH)
)

print("Filas:", df_raw.count())
print("Columnas:", len(df_raw.columns))
df_raw.select("loan_status").groupBy("loan_status").count().orderBy(col("count").desc()).show(10, truncate=False)

Filas: 2260701
Columnas: 151
+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1076751|
|Current                                            |878317 |
|Charged Off                                        |268559 |
|Late (31-120 days)                                 |21467  |
|In Grace Period                                    |8436   |
|Late (16-30 days)                                  |4349   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |40     |
|NULL                                               |33     |
+---------------------------------------------------+-------+



## 6.3 Creación de la etiqueta (label)

Se construye la variable objetivo `label` para clasificación binaria:
- 0 = No Default (Fully Paid)
- 1 = Default (Charged Off)

Se filtran únicamente estas dos clases para mantener coherencia con el modelado.

In [6]:
from pyspark.sql.functions import when, upper, trim

# Caso estándar LendingClub: loan_status
if "loan_status" in df_raw.columns:
    df = df_raw.filter(col("loan_status").isin(["Fully Paid", "Charged Off"]))
    df = df.withColumn("label", when(col("loan_status") == "Charged Off", 1).otherwise(0))
else:
    # Plan B: si existe columna "default"
    if "default" in df_raw.columns:
        df = df_raw.withColumn(
            "label",
            when(upper(trim(col("default"))).isin("1", "Y", "YES", "TRUE", "T"), 1)
            .when(upper(trim(col("default"))).isin("0", "N", "NO", "FALSE", "F"), 0)
            .otherwise(None)
        ).filter(col("label").isNotNull())
    else:
        raise ValueError("No se encontró ni 'loan_status' ni 'default' para crear label.")

df.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|    1| 268559|
|    0|1076751|
+-----+-------+



## 6.4 Selección de variables, limpieza y balanceo (weight)

Se seleccionan variables numéricas y categóricas, se corrigen tipos (cast a double) y se crea `weight` para balancear clases (equivalente a `class_weight="balanced"` en sklearn).

In [7]:
from pyspark.sql.functions import regexp_replace

#  Lista de features (puedes ajustar, pero estas son estables y suelen existir)
num_cols = [
    "loan_amnt", "funded_amnt", "funded_amnt_inv",
    "int_rate", "installment",
    "annual_inc", "dti",
    "fico_range_low", "fico_range_high"
]

cat_cols = [
    "term", "grade", "home_ownership", "verification_status", "purpose", "addr_state"
]

# Solo quedarnos con columnas que existan (evita errores si alguna no está)
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

keep_cols = num_cols + cat_cols + ["label"]
if "loan_status" in df.columns:
    keep_cols += ["loan_status"]

df = df.select(*keep_cols)

# ---- CAST ROBUSTO A DOUBLE (arregla errores de "string not supported") ----
for c in num_cols:
    # quitar %, comas, espacios; convertir vacío a null
    df = df.withColumn(c, regexp_replace(col(c).cast("string"), "%", ""))
    df = df.withColumn(c, regexp_replace(col(c).cast("string"), ",", ""))
    df = df.withColumn(c, when(trim(col(c)) == "", None).otherwise(col(c)))
    df = df.withColumn(c, col(c).cast("double"))

# ---- PESOS PARA BALANCEO ----
counts = df.groupBy("label").count().collect()
count_dict = {row["label"]: row["count"] for row in counts}
n0 = count_dict.get(0, 1)
n1 = count_dict.get(1, 1)
total = n0 + n1

w0 = total / (2.0 * n0)
w1 = total / (2.0 * n1)

df = df.withColumn("weight", when(col("label") == 1, w1).otherwise(w0))

print("n0:", n0, "n1:", n1)
print("w0:", w0, "w1:", w1)
df.select("label", "weight").groupBy("label", "weight").count().show()

n0: 1076751 n1: 268559
w0: 0.6247080337050999 w1: 2.504682397536482
+-----+------------------+-------+
|label|            weight|  count|
+-----+------------------+-------+
|    0|0.6247080337050999|1076751|
|    1| 2.504682397536482| 268559|
+-----+------------------+-------+



## 6.5 Construcción del pipeline

Se define un pipeline con:
- Indexado y OneHot para categóricas
- Imputación para numéricas
- Ensamble de características (VectorAssembler)
- Modelo de clasificación con `weightCol`

In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.classification import LogisticRegression

# Indexers para categóricas
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

# OneHot para categóricas indexadas
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

# Imputer para numéricas
imputer = Imputer(
    inputCols=num_cols,
    outputCols=[f"{c}_imp" for c in num_cols]
)

# VectorAssembler final
assembler_inputs = [f"{c}_imp" for c in num_cols] + [f"{c}_ohe" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

#  Modelo robusto para desbalance: Logistic Regression con weightCol
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=50
)

pipeline = Pipeline(stages=indexers + [encoder, imputer, assembler, lr])

## 6.6 Entrenamiento y predicción

Se divide el dataset en entrenamiento y prueba, se entrena el pipeline y se generan predicciones.

In [9]:
import time

df = df.cache()

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

t0 = time.time()
spark_model = pipeline.fit(train_df)
train_time = time.time() - t0

t1 = time.time()
pred = spark_model.transform(test_df)
pred_time = time.time() - t1

print(f"Tiempo entrenamiento: {train_time:.2f} s")
print(f"Tiempo predicción:    {pred_time:.2f} s")

pred.select("label", "prediction", "probability").show(5, truncate=False)

Tiempo entrenamiento: 137.56 s
Tiempo predicción:    0.25 s
+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.5477333739311461,0.4522666260688539] |
|0    |0.0       |[0.8613445032144256,0.13865549678557443]|
|0    |0.0       |[0.8619438093722148,0.13805619062778518]|
|1    |0.0       |[0.5103916068409683,0.4896083931590317] |
|0    |0.0       |[0.6003581098129126,0.3996418901870874] |
+-----+----------+----------------------------------------+
only showing top 5 rows


## 6.7 Evaluación de métricas y ajuste de threshold

Debido al desbalance de clases, el threshold por defecto (0.5) puede producir recall muy bajo.
Se evalúan métricas usando un threshold ajustable basado en la probabilidad de clase 1.

In [11]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when

# probabilidad clase 1 (Default)
pred = pred.withColumn("p1", vector_to_array(col("probability"))[1])

thresholds = [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]

def eval_threshold(TH):
    tmp = pred.withColumn("pred_thr", when(col("p1") >= TH, 1.0).otherwise(0.0))
    mdf = tmp.select(col("label"), col("pred_thr").alias("prediction"))

    tp = mdf.filter((col("label")==1) & (col("prediction")==1)).count()
    tn = mdf.filter((col("label")==0) & (col("prediction")==0)).count()
    fp = mdf.filter((col("label")==0) & (col("prediction")==1)).count()
    fn = mdf.filter((col("label")==1) & (col("prediction")==0)).count()

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    acc = (tp+tn) / (tp+tn+fp+fn) if (tp+tn+fp+fn) else 0

    return (TH, acc, precision, recall, f1, tp, fp, fn, tn)

results = [eval_threshold(th) for th in thresholds]

print("TH | Acc | Prec | Rec | F1 | TP FP FN TN")
for r in results:
    print(f"{r[0]:.2f} | {r[1]:.4f} | {r[2]:.4f} | {r[3]:.4f} | {r[4]:.4f} | {r[5]} {r[6]} {r[7]} {r[8]}")

# elegir threshold final
TH = 0.15
pred = pred.withColumn("pred_thr", when(col("p1") >= TH, 1.0).otherwise(0.0))

TH | Acc | Prec | Rec | F1 | TP FP FN TN
0.50 | 0.6395 | 0.3116 | 0.6674 | 0.4248 | 35772 79032 17826 136064
0.30 | 0.3889 | 0.2376 | 0.9341 | 0.3788 | 50066 160663 3532 54433
0.20 | 0.2849 | 0.2158 | 0.9815 | 0.3538 | 52608 191153 990 23943
0.15 | 0.2319 | 0.2056 | 0.9954 | 0.3408 | 53352 206150 246 8946
0.10 | 0.2034 | 0.2002 | 0.9994 | 0.3336 | 53564 213997 34 1099
0.05 | 0.1996 | 0.1995 | 0.9999 | 0.3326 | 53592 215057 6 39


## 6.8 ROC-AUC

Se calcula el área bajo la curva ROC (AUC), una métrica robusta para clasificación binaria.

In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator_auc.evaluate(pred)
print("ROC-AUC:", auc)

ROC-AUC: 0.7082742258806913
